In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sys
from neuralop.models import TFNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.utils import count_model_params, get_project_root
from neuralop import LpLoss, H1Loss
from pathlib import Path

from heat import load_heat_1dtime

device = "cuda"

n_train = 40000
n_val = 500

save_path = Path("heat_data") / f"train{n_train}_test{n_val}"

train_loader, val_loader, data_processor = load_heat_1dtime(
    path=save_path,
    batch_size=64,
    test_batch_size=n_val,
)

val_loaders = {}
val_loaders[100] = val_loader

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
)
model = model.to(device)

# Count and display the number of parameters
n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()

optimizer = AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)  # weight_decay can be tuned
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=1000, eta_min=1e-8
)

l2loss = LpLoss(d=2, p=2)  # L2 loss for function values
h1loss = H1Loss(d=2)  # H1 loss includes gradient information

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}

print("\n### MODEL ###\n", model)
print("\n### OPTIMIZER ###\n", optimizer)
print("\n### SCHEDULER ###\n", scheduler)
print("\n### LOSSES ###")
print(f"\n * Train: {train_loss}")
print(f"\n * Test: {eval_losses}")
sys.stdout.flush()

trainer = Trainer(
    model=model,
    n_epochs=1000,
    device=device,
    data_processor=data_processor().to(device),
    wandb_log=True,  # Disable Weights & Biases logging for this tutorial
    eval_interval=20,  # Evaluate every 5 epochs
    use_distributed=False,  # Single GPU/CPU training
    verbose=True,  # Print training progress
)

trainer.train(
    train_loader=train_loader,
    test_loaders=val_loaders,
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    eval_losses=eval_losses,
)

In [ ]:
save_dir = Path("checkpoints")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "tfno_heat_1d_40500.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to {model_path}")

In [ ]:
# Testing with randomly drawn parameters a

import torch
import numpy as np
import matplotlib.pyplot as plt


def heat_solution_closed_form_grid(x_grid, t_grid, a_coeffs, alpha=0.25):
    x = x_grid[None, None, :]  # (1, 1, Gx)
    t = t_grid[None, :, None]  # (1, Gt, 1)

    N, L = a_coeffs.shape
    l = torch.arange(1, L + 1, device=a_coeffs.device, dtype=a_coeffs.dtype)  # (L,)

    a = a_coeffs[:, :, None, None]  # (N, L, 1, 1)
    l = l[None, :, None, None]  # (1, L, 1, 1)

    decay = torch.exp(-(alpha**2) * (l * torch.pi) ** 2 * t)  # (1, L, Gt, 1)
    sines = torch.sin(l * torch.pi * x)  # (1, L, 1, Gx)

    u = torch.sum(a * decay * sines, dim=1)  # (N, Gt, Gx)
    return u.unsqueeze(1)  # (N, 1, Gt, Gx)


def build_initial_condition_heat(a_coeffs, x_grid, Gt):
    N, n_modes = a_coeffs.shape
    Gx = x_grid.numel()

    u0 = torch.zeros(N, Gx, device=a_coeffs.device, dtype=a_coeffs.dtype)

    for l in range(n_modes):
        u0 += a_coeffs[:, l].unsqueeze(1) * torch.sin((l + 1) * torch.pi * x_grid)

    u0[:, 0] = 0.0
    u0[:, -1] = 0.0

    return u0.unsqueeze(1).unsqueeze(2).repeat(1, 1, Gt, 1)  # (N, 1, Gt, Gx)


device = "cuda"
N_eval = 10**4
n_modes = 9
Gx = 100
Gt = 100
bs = 32

# grids
x_grid = torch.linspace(0, 1, Gx, device=device)
t_grid = torch.linspace(0, 1, Gt, device=device)

# model
from neuralop.models import FNO, TFNO

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)
model.load_state_dict(
    torch.load("checkpoints/tfno_heat_1d.pt", map_location=device)
)  # Add weights_only=False for FNO
model.eval()

errs = []

with torch.no_grad():
    for i in range(0, N_eval, bs):
        cur_bs = min(bs, N_eval - i)

        # generate coefficients per batch (no need to store all 1e4 on GPU)
        a_coeffs = 2.0 * torch.rand(cur_bs, n_modes, device=device) - 1.0

        # reference + input
        u_ref = heat_solution_closed_form_grid(x_grid, t_grid, a_coeffs, alpha=0.25)
        u0_eval = build_initial_condition_heat(a_coeffs, x_grid, Gt)

        # prediction
        u_pred = model(u0_eval)

        # relative L2 error per sample
        err = torch.norm(u_ref - u_pred, dim=(-1, -2)) / torch.norm(u_ref, dim=(-1, -2))
        errs.append(err.detach().cpu())

err = torch.cat(errs, dim=0).numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(err, vert=False, widths=0.5, patch_artist=True)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title("Boxplot of relative $L^2$ Errors for the 1D Heat equation", fontsize=13)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Use this if you have a precomputed set of parameters a for comparability

import os
import torch
import numpy as np
import matplotlib.pyplot as plt


def heat_solution_closed_form_grid(x_grid, t_grid, a_coeffs, alpha=0.25):
    x = x_grid[None, None, :]
    t = t_grid[None, :, None]

    N, L = a_coeffs.shape
    l = torch.arange(1, L + 1, device=a_coeffs.device, dtype=a_coeffs.dtype)

    a = a_coeffs[:, :, None, None]
    l = l[None, :, None, None]

    decay = torch.exp(-(alpha**2) * (l * torch.pi) ** 2 * t)
    sines = torch.sin(l * torch.pi * x)

    u = torch.sum(a * decay * sines, dim=1)
    return u.unsqueeze(1)


def build_initial_condition_heat(a_coeffs, x_grid, Gt):
    N, n_modes = a_coeffs.shape
    Gx = x_grid.numel()

    u0 = torch.zeros(N, Gx, device=a_coeffs.device, dtype=a_coeffs.dtype)

    for l in range(n_modes):
        u0 += a_coeffs[:, l].unsqueeze(1) * torch.sin((l + 1) * torch.pi * x_grid)

    u0[:, 0] = 0.0
    u0[:, -1] = 0.0

    return u0.unsqueeze(1).unsqueeze(2).repeat(1, 1, Gt, 1)


device = "cuda"
bs = 32
Gx = 100
Gt = 100

currentdir = os.getcwd()
parentdir = os.path.dirname(currentdir)

testset_path = os.path.join(parentdir, "heat_testset_10000.npy")
checkpoint_path = "checkpoints/tfno_heat_1d_40500.pt"

a_test = np.load(testset_path)
a_test = torch.tensor(a_test, dtype=torch.float32, device=device)
N_eval, n_modes = a_test.shape

x_grid = torch.linspace(0, 1, Gx, device=device)
t_grid = torch.linspace(0, 1, Gt, device=device)

from neuralop.models import TFNO

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

errs = []

with torch.no_grad():
    for i in range(0, N_eval, bs):
        a_coeffs = a_test[i : i + bs]

        u_ref = heat_solution_closed_form_grid(x_grid, t_grid, a_coeffs, alpha=0.25)
        u0_eval = build_initial_condition_heat(a_coeffs, x_grid, Gt)
        u_pred = model(u0_eval)

        err = torch.norm(u_ref - u_pred, dim=(-1, -2)) / torch.norm(u_ref, dim=(-1, -2))
        errs.append(err.cpu())

err = torch.cat(errs, dim=0).numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(err, vert=False, widths=0.5, patch_artist=True)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title("Boxplot of relative $L^2$ Errors for the 1D Heat equation", fontsize=13)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()